# Glider / CTD 3D Curtain Plots — Barkley Sound

Plots glider tracks and CTD casts in 3D (Longitude, Latitude, Depth), colored by a variable of
interest (e.g. temperature, salinity, chlorophyll, oxygen, ...), draped over Barkley Sound
bathymetry (same look as the AOOS Ocean Data Explorer curtain plots).

- **Gliders** move through lon/lat while diving — rendered as a dense 3D scatter "ribbon".
- **CTD casts** are stationary (fixed lon/lat, only depth varies) — rendered as a **standard 2D
  profile** (variable vs. depth, surface at top) instead of 3D. A cast doesn't move horizontally,
  so a 3D curtain would misleadingly imply spatial extent it doesn't have; where a cast was taken
  is shown by its position on the map (outside this notebook), not by this plot.

Column names in real files vary a lot (`Lon`/`Longitude`/`lng`, different casing, longitude in
0–360° vs −180–180°, ...) — the loaders below auto-detect and standardize all of this. See
Section 4.

Edit the `CONFIG` cell below and re-run the notebook.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# rioxarray / xarray are only needed if you load real GeoTIFF/NetCDF files.
# They're imported lazily inside the loader functions so this notebook still runs without them.


## 2. Configuration

This is the only cell you should need to edit for a new dataset or region.

**Glider / CTD data** — both use the *same* 4-column schema (Lon, Lat, Depth, variable), so they
share one loader (`load_platform_data()`). For a CTD cast, Lon/Lat just happen to be ~constant
across rows since the instrument isn't moving.

- `USE_SAMPLE_DATA`: `True` to generate a synthetic demo track/cast, `False` to load a real file.
- `DATA_PATH` / `FILE_TYPE`: path to your CSV or NetCDF file (`"csv"` or `"netcdf"`).
- `COLUMN_MAP`: maps standard fields (`lon`, `lat`, `depth`, `variable`) to actual column names.
  **`lon`, `lat`, and `depth` are auto-detected** — leave them `None` and the loader recognizes
  common spellings case-insensitively (`Lon`/`LONGITUDE`/`lng`/`x`, `Lat`/`LATITUDE`/`y`,
  `Depth`/`z`/`depth_m`). Only set them explicitly if your file uses something the aliases don't
  cover. `variable` always needs to be set explicitly (it's arbitrary per dataset) but is matched
  case-insensitively too. Longitude is also automatically standardized to −180–180°, whether the
  source file uses that convention or 0–360°.
- `VARIABLE_LABEL`: axis/colorbar title text.
- `COLOR_SCALE` (glider only): continuous Plotly colorscale for the 3D curtain's color axis.
- `LINE_COLOR` (CTD only): a single hex color for the 2D profile line — no colorbar needed since
  the variable is already an axis, not a color encoding, on a one-series 2D plot.
- `DEPTH_POSITIVE_DOWN`: leave `True` if depth increases downward as a positive number (the
  normal glider/CTD convention).

**Bathymetry** — a gridded seafloor surface draped underneath the curtain, clipped and
downsampled to the area around whatever's being plotted. Same auto-detection applies, plus
`elevation` is recognized as a depth alias (the standard GEBCO variable name).

- `MAX_GRID_SIZE`: max points per axis after downsampling, for smooth 3D rendering.
- `BUFFER_DEG`: how far past the data's bounding box to include seafloor, in degrees.

We're not correcting for a bathymetry/CTD vertical-datum offset since gliders in Barkley Sound
don't reach the bottom anyway — the seafloor is for visual/spatial context only.

In [ ]:
CONFIG = {
    "GLIDER": {
        "USE_SAMPLE_DATA": False,
        "DATA_PATH": "path/to/glider_data.csv",
        "FILE_TYPE": "csv",
        "COLUMN_MAP": {"lon": None, "lat": None, "depth": None, "variable": "Temperature"},
        "VARIABLE_LABEL": "Temperature (\u00b0C)",
        "COLOR_SCALE": "Thermal",
        "DEPTH_POSITIVE_DOWN": True,
        "MARKER_SIZE": 4,
        "USE_BATHYMETRY": True,   # Barkley Sound sample track -> same region as CONFIG["BATHYMETRY"]
    },
    "CTD": {
        "USE_SAMPLE_DATA": False,
        "DATA_PATH": "NE_San_Diego_Trough_Aug_2022.csv",
        "FILE_TYPE": "csv",
        # lon/lat auto-detect via the "_dec" aliases (this file uses "Lat_Dec"/"Lon_Dec").
        # "Salt2" is the preferred salinity sensor for this cast (Salt1/Salt2 are the cast's
        # two conductivity sensors; SaltAve_Corr etc. are also present but Salt2 is preferred).
        "COLUMN_MAP": {"lon": None, "lat": None, "depth": None, "variable": "Salt2"},
        "VARIABLE_LABEL": "Salinity (PSU)",
        "LINE_COLOR": "#1b6ca8",
        "DEPTH_POSITIVE_DOWN": True,
        "MARKER_SIZE": 4,
    },
    "BATHYMETRY": {
        "USE_SAMPLE_BATHY": False,
        "DATA_PATH": "Barkley_Sound_Bathymetry.nc",
        "FILE_TYPE": "netcdf",           # "geotiff" or "netcdf"
        "COLUMN_MAP": {"lon": None, "lat": None, "depth": "elevation"},
        # GEBCO's "elevation" is already negative-down / positive-up (sea depth is negative),
        # which is exactly the convention this notebook plots in \u2014 so no sign flip needed.
        "DEPTH_POSITIVE_DOWN": False,
        "COLOR_SCALE": "gray",
        "OPACITY": 0.85,
        "BUFFER_DEG": 0.02,              # padding around track/cast footprint, in degrees
        "MAX_GRID_SIZE": 200,            # downsample target per axis, for smooth 3D rendering
    },

    # Barkley Sound, BC bounding box \u2014 used only for the synthetic sample glider/CTD data below
    "REGION": {"lon_range": (-125.35, -125.05), "lat_range": (48.83, 48.95)},
}


## 3. Sample data generators (used when `USE_SAMPLE_DATA` / `USE_SAMPLE_BATHY` is `True`)

In [ ]:
def generate_sample_glider_data(num_points=500, variable_col="Temperature",
                                 lon_range=None, lat_range=None, max_depth=150):
    """Simulate a sawtooth glider track with a synthetic variable field, over the sample region."""
    lon_range = lon_range or CONFIG["REGION"]["lon_range"]
    lat_range = lat_range or CONFIG["REGION"]["lat_range"]
    time = np.linspace(0, 10, num_points)

    lon = lon_range[0] + (lon_range[1] - lon_range[0]) * (time / time.max())
    lat = lat_range[0] + (lat_range[1] - lat_range[0]) * (time / time.max())

    # Sawtooth depth profile, positive-down
    depth = (max_depth / 2) * (1 + np.sin(2 * np.pi * time))

    # Synthetic variable decreasing with depth, with noise
    variable = 12 - (depth * 0.02) + np.random.normal(0, 0.3, num_points)

    return pd.DataFrame({"Longitude": lon, "Latitude": lat, "Depth": depth, variable_col: variable})


def generate_sample_ctd_data(num_points=60, variable_col="Salinity",
                              lon=None, lat=None, max_depth=150):
    """Simulate a single stationary CTD cast \u2014 fixed (lon, lat), variable varying with depth."""
    region = CONFIG["REGION"]
    lon = lon if lon is not None else np.mean(region["lon_range"])
    lat = lat if lat is not None else np.mean(region["lat_range"])

    depth = np.linspace(0, max_depth, num_points)
    variable = 30 + (depth * 0.01) + np.random.normal(0, 0.1, num_points)  # e.g. salinity ~increasing with depth

    return pd.DataFrame({
        "Longitude": np.full(num_points, lon),
        "Latitude": np.full(num_points, lat),
        "Depth": depth,
        variable_col: variable,
    })


def generate_sample_bathymetry(lon_range=None, lat_range=None, grid_size=150, max_depth=180):
    """Synthetic bowl-shaped seafloor roughly resembling a sound/inlet, for demo purposes."""
    lon_range = lon_range or CONFIG["REGION"]["lon_range"]
    lat_range = lat_range or CONFIG["REGION"]["lat_range"]

    lon = np.linspace(*lon_range, grid_size)
    lat = np.linspace(*lat_range, grid_size)
    lon_grid, lat_grid = np.meshgrid(lon, lat)

    lon_c, lat_c = np.mean(lon_range), np.mean(lat_range)
    r = np.sqrt(((lon_grid - lon_c) / (lon_range[1] - lon_range[0])) ** 2 +
                ((lat_grid - lat_c) / (lat_range[1] - lat_range[0])) ** 2)
    depth = -max_depth * np.clip(1 - r, 0, 1) ** 0.7  # shallower at the edges, deepest mid-sound

    return {"lon": lon, "lat": lat, "depth": depth}


## 4. Column & coordinate standardization

Real datasets name things inconsistently — `Lon` vs `Longitude` vs `lng` vs `x`, different
casing, longitude in 0–360° vs −180–180°. These helpers let every loader below accept any of
that without needing exact column names.

- `resolve_column()` finds the right column for a standard field (`lon`, `lat`, `depth`, ...) by
  trying a list of known aliases, case-insensitively. An explicit override (from `COLUMN_MAP`)
  always wins if given.
- `standardize_longitude()` converts any longitude convention to −180–180° with one formula, so
  it works whether the source was already −180–180 or 0–360 — no need to know which in advance.

In [ ]:
STANDARD_ALIASES = {
    "lon": ["lon", "longitude", "long", "lng", "x", "lon_dec", "decimallongitude"],
    "lat": ["lat", "latitude", "y", "lat_dec", "decimallatitude"],
    "depth": ["depth", "z", "depth_m", "z_pos"],
}
# Note: "variable" (e.g. salinity) is never looked up via an alias list here -- it's always
# whatever exact column you set in COLUMN_MAP["variable"] (matched case-insensitively). For
# NE_San_Diego_Trough_Aug_2022.csv that's "Salt2" -- see CONFIG["CTD"] below. Salt1/Salt2 are
# the cast's two conductivity sensors; Salt2 is the preferred/selected one for this cruise.
BATHY_ALIASES = {
    **STANDARD_ALIASES,
    "depth": ["depth", "elevation", "elev", "z", "bathymetry", "topo"],
}


def resolve_column(available, standard_key, aliases, override=None):
    """Find the actual column/variable name for a standard field, matching case-insensitively.

    `override` (e.g. from CONFIG's COLUMN_MAP) is tried first if given; otherwise falls back to
    the alias list for `standard_key`.
    """
    lookup = {str(c).lower(): c for c in available}
    candidates = [override] if override else aliases.get(standard_key, [standard_key])
    for candidate in candidates:
        if candidate and str(candidate).lower() in lookup:
            return lookup[str(candidate).lower()]
    raise KeyError(
        f"Could not find a column for '{standard_key}' (tried {candidates}). "
        f"Available columns: {list(available)}. "
        f"Set COLUMN_MAP['{standard_key}'] explicitly if your file uses a different name."
    )


def standardize_longitude(lon):
    """Convert longitude to the standard -180-180 range, regardless of whether the source used
    that convention or 0-360. Idempotent \u2014 safe to call on already-standard data."""
    lon = np.asarray(lon, dtype=float)
    return ((lon + 180) % 360) - 180


## 5. Real data loaders (used when `USE_SAMPLE_DATA` / `USE_SAMPLE_BATHY` is `False`)

`load_platform_data()` handles both gliders and CTD casts — same 4-column schema either way.

In [ ]:
def load_platform_data(path, file_type, column_map):
    """Load a real glider track or CTD cast and standardize it to Longitude/Latitude/Depth/<variable>.

    Works for both platform types: a CTD cast is just a file where lon/lat barely vary.
    `column_map` maps standard keys ("lon", "lat", "depth", "variable") to actual column names.
    lon/lat/depth are auto-detected (case-insensitive, common aliases) if left as None in
    column_map \u2014 only "variable" must be given, since it's arbitrary per dataset.
    """
    if file_type == "csv":
        raw = pd.read_csv(path)
    elif file_type in ("netcdf", "nc"):
        import xarray as xr
        raw = xr.open_dataset(path).to_dataframe().reset_index()
    else:
        raise ValueError(f"Unsupported FILE_TYPE: {file_type!r} (expected 'csv' or 'netcdf')")

    lon_col = resolve_column(raw.columns, "lon", STANDARD_ALIASES, column_map.get("lon"))
    lat_col = resolve_column(raw.columns, "lat", STANDARD_ALIASES, column_map.get("lat"))
    depth_col = resolve_column(raw.columns, "depth", STANDARD_ALIASES, column_map.get("depth"))
    if not column_map.get("variable"):
        raise KeyError("COLUMN_MAP['variable'] must be set \u2014 it can't be auto-detected.")
    var_col = resolve_column(raw.columns, "variable", {}, column_map["variable"])

    df = raw[[lon_col, lat_col, depth_col, var_col]].copy()
    df.columns = ["Longitude", "Latitude", "Depth", column_map["variable"]]
    df["Longitude"] = standardize_longitude(df["Longitude"])
    df = df.dropna().reset_index(drop=True)
    return df


def load_bathymetry(path, file_type, column_map=None, depth_positive_down=False):
    """Load a gridded bathymetry/DEM file, standardized to lon (1D), lat (1D), depth (2D).

    lon/lat/depth names are auto-detected (case-insensitive, common aliases incl. "elevation")
    unless overridden in column_map.

    `depth_positive_down`: set True only if your file's depth variable is stored as a positive
    number that increases downward. GEBCO-style "elevation" grids (like Barkley_Sound_Bathymetry.nc)
    are already negative-down / positive-up (sea depth negative, land positive) \u2014 leave False.
    """
    column_map = column_map or {}
    if file_type in ("geotiff", "tif", "tiff"):
        import rioxarray
        da = rioxarray.open_rasterio(path).squeeze()
        lon, lat, depth = da["x"].values, da["y"].values, da.values
    elif file_type in ("netcdf", "nc"):
        import xarray as xr
        ds = xr.open_dataset(path)
        available = list(ds.coords) + list(ds.data_vars)
        lon_col = resolve_column(available, "lon", BATHY_ALIASES, column_map.get("lon"))
        lat_col = resolve_column(available, "lat", BATHY_ALIASES, column_map.get("lat"))
        depth_col = resolve_column(available, "depth", BATHY_ALIASES, column_map.get("depth"))
        lon, lat, depth = ds[lon_col].values, ds[lat_col].values, ds[depth_col].values
    else:
        raise ValueError(f"Unsupported bathymetry FILE_TYPE: {file_type!r} (expected 'geotiff' or 'netcdf')")

    lon, lat, depth = np.asarray(lon, dtype=float), np.asarray(lat, dtype=float), np.asarray(depth)

    # Standardize longitude to -180/180 and re-sort ascending (0-360 grids can wrap around the
    # seam after conversion, e.g. [..., 358, 359, 0, 1, 2, ...] -> [..., -2, -1, 0, 1, 2, ...])
    lon = standardize_longitude(lon)
    lon_order = np.argsort(lon)
    lon, depth = lon[lon_order], depth[:, lon_order]

    if lat[0] > lat[-1]:  # rasters are often north-up / descending; make ascending for consistent indexing
        lat, depth = lat[::-1], depth[::-1, :]
    if depth_positive_down:
        depth = -np.abs(depth)

    return {"lon": lon, "lat": lat, "depth": depth}


## 6. Bathymetry helpers

Clips the (possibly large, full-region) bathymetry grid down to the footprint of whatever track
or cast is being plotted, plus a buffer, and downsamples it so the 3D surface renders smoothly.
Raises a clear error (rather than silently plotting nothing) if the grid doesn't reach the data.

In [ ]:
def clip_and_decimate_bathymetry(bathy, lon_bounds, lat_bounds, buffer_deg=0.02, max_grid_size=200):
    """Crop a bathymetry grid to the area around some data (+ buffer) and thin it for fast 3D rendering."""
    lon, lat, depth = bathy["lon"], bathy["lat"], bathy["depth"]

    lon_mask = (lon >= lon_bounds[0] - buffer_deg) & (lon <= lon_bounds[1] + buffer_deg)
    lat_mask = (lat >= lat_bounds[0] - buffer_deg) & (lat <= lat_bounds[1] + buffer_deg)

    lon_clip = lon[lon_mask]
    lat_clip = lat[lat_mask]

    if len(lon_clip) == 0 or len(lat_clip) == 0:
        raise ValueError(
            "Bathymetry grid does not cover this data's footprint.\n"
            f"  Bathymetry covers lon [{lon.min():.3f}, {lon.max():.3f}], lat [{lat.min():.3f}, {lat.max():.3f}]\n"
            f"  Data (+ buffer) needs lon [{lon_bounds[0]-buffer_deg:.3f}, {lon_bounds[1]+buffer_deg:.3f}], "
            f"lat [{lat_bounds[0]-buffer_deg:.3f}, {lat_bounds[1]+buffer_deg:.3f}]\n"
            "  Get a bathymetry file that covers the data's actual location, or plot without bathymetry (bathymetry=None)."
        )

    depth_clip = depth[np.ix_(lat_mask, lon_mask)]

    lon_step = max(1, len(lon_clip) // max_grid_size)
    lat_step = max(1, len(lat_clip) // max_grid_size)

    return {
        "lon": lon_clip[::lon_step],
        "lat": lat_clip[::lat_step],
        "depth": depth_clip[::lat_step, ::lon_step],
    }


def add_bathymetry_surface(fig, bathy, lon_bounds, lat_bounds, buffer_deg=0.02,
                            max_grid_size=200, colorscale="gray", opacity=0.85):
    """Add a shaded seafloor Surface trace to an existing 3D figure, clipped to the data's footprint."""
    patch = clip_and_decimate_bathymetry(bathy, lon_bounds, lat_bounds, buffer_deg, max_grid_size)
    fig.add_trace(go.Surface(
        x=patch["lon"], y=patch["lat"], z=patch["depth"],
        colorscale=colorscale, showscale=False, opacity=opacity,
        lighting=dict(ambient=0.6, diffuse=0.8, specular=0.1),
        name="Seafloor",
    ))
    return fig


### Bathymetry coverage check (diagnostic)

Before trusting a bathymetry file as the curtain-plot basemap, it's worth seeing what area it
actually covers — real-world grids get clipped wrong, or have gaps. `plot_bathymetry_extent()`
draws the grid on a simple coastline basemap (`cartopy`), colored with an intuitive land/sea
diverging scale (`cmocean.cm.topo`: blue = underwater depth, green/tan/brown = land elevation,
centered at 0), and can outline a region of interest so you can see at a glance whether the file
actually reaches it. These are optional extra dependencies, only needed for this diagnostic.

In [ ]:
def plot_bathymetry_extent(bathy, region_box=None, region_label=None, mask_exact_zero=False,
                            max_grid_size=800, colormap=None, figsize=(9, 7.5)):
    """Quick 2D overview map of a bathymetry grid's real extent and coloring, with coastlines
    for geographic context \u2014 a sanity check before using it as a curtain-plot basemap.

    `region_box`: optional ((lon_min, lon_max), (lat_min, lat_max)) to outline a target area of
    interest (e.g. your study region) on the map, labeled with `region_label`.
    `mask_exact_zero`: some grids use exactly 0.0 as a nodata/fill value instead of NaN (seen in
    the northern part of Barkley_Sound_Bathymetry.nc) \u2014 set True to mask those for display.
    Leave False unless you've confirmed 0.0 isn't real data in your file.
    """
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    import matplotlib.patches as mpatches
    if colormap is None:
        try:
            import cmocean
            colormap = cmocean.cm.topo
        except ImportError:
            colormap = "BrBG_r"  # reasonable land/sea diverging fallback

    lon, lat, depth = bathy["lon"], bathy["lat"], bathy["depth"].astype(float)
    if mask_exact_zero:
        depth = depth.copy()
        depth[depth == 0.0] = np.nan

    lon_step = max(1, len(lon) // max_grid_size)
    lat_step = max(1, len(lat) // max_grid_size)
    lon_d, lat_d, depth_d = lon[::lon_step], lat[::lat_step], depth[::lat_step, ::lon_step]

    lon_min, lon_max, lat_min, lat_max = lon.min(), lon.max(), lat.min(), lat.max()
    if region_box is not None:
        (rlon_min, rlon_max), (rlat_min, rlat_max) = region_box
        lon_min, lon_max = min(lon_min, rlon_min) - 0.3, max(lon_max, rlon_max) + 0.3
        lat_min, lat_max = min(lat_min, rlat_min) - 0.3, max(lat_max, rlat_max) + 0.3

    fig = plt.figure(figsize=figsize)
    ax = fig.add_axes([0.09, 0.10, 0.72, 0.80], projection=ccrs.PlateCarree())
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    vmax = max(1, float(np.nanmax(depth_d)))
    norm = mcolors.TwoSlopeNorm(vmin=float(np.nanmin(depth_d)), vcenter=0, vmax=vmax)
    img = ax.imshow(depth_d, extent=[lon_d.min(), lon_d.max(), lat_d.min(), lat_d.max()],
                     origin="lower", transform=ccrs.PlateCarree(), cmap=colormap, norm=norm,
                     interpolation="antialiased")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, edgecolor="black")
    gl = ax.gridlines(draw_labels=True, alpha=0.3, linewidth=0.4)
    gl.top_labels = False
    gl.right_labels = False

    cax = fig.add_axes([0.83, 0.15, 0.03, 0.65])
    cb = fig.colorbar(img, cax=cax)
    cb.set_label("Elevation (m)  \u2014  land above 0, sea depth below 0")

    if region_box is not None:
        rect = mpatches.Rectangle((rlon_min, rlat_min), rlon_max - rlon_min, rlat_max - rlat_min,
                                   transform=ccrs.PlateCarree(), fill=False, edgecolor="red",
                                   linewidth=2, zorder=6)
        ax.add_patch(rect)
        if region_label:
            ax.annotate(region_label, xy=(rlon_min, rlat_max),
                        xytext=(lon_min + 0.4 * (lon_max - lon_min), lat_min + 0.05 * (lat_max - lat_min)),
                        textcoords=ccrs.PlateCarree()._as_mpl_transform(ax),
                        color="red", fontsize=9, ha="left", va="bottom",
                        arrowprops=dict(arrowstyle="->", color="red", lw=1.2))

    fig.suptitle("Bathymetry coverage extent", fontsize=13, y=0.96)
    plt.show()
    return fig




## 7. Plot functions

In [ ]:
def plot_glider_curtain(df, variable_col, variable_label=None, color_scale="Thermal",
                         title=None, marker_size=4, bathymetry=None,
                         bathy_buffer_deg=0.02, bathy_max_grid_size=200,
                         bathy_colorscale="gray", bathy_opacity=0.85):
    """Build an interactive 3D 'curtain' plot of a track/cast colored by one variable,
    optionally draped over a clipped patch of bathymetry."""
    variable_label = variable_label or variable_col
    title = title or f"3D {variable_label} Curtain Plot"

    fig = go.Figure()

    if bathymetry is not None:
        lon_bounds = (df["Longitude"].min(), df["Longitude"].max())
        lat_bounds = (df["Latitude"].min(), df["Latitude"].max())
        add_bathymetry_surface(fig, bathymetry, lon_bounds, lat_bounds,
                                buffer_deg=bathy_buffer_deg, max_grid_size=bathy_max_grid_size,
                                colorscale=bathy_colorscale, opacity=bathy_opacity)

    # Small, dense markers make the scatter read as a continuous ribbon/curtain
    fig.add_trace(go.Scatter3d(
        x=df["Longitude"], y=df["Latitude"], z=df["Depth"],
        mode="markers",
        marker=dict(size=marker_size, color=df[variable_col], colorscale=color_scale,
                    colorbar=dict(title=variable_label)),
        name=variable_label,
    ))

    fig.update_layout(
        title=title,
        scene=dict(xaxis_title="Longitude", yaxis_title="Latitude", zaxis_title="Depth (m)"),
    )
    return fig


def plot_ctd_profile(df, variable_col, variable_label=None, title=None,
                      line_color="#1b6ca8", line_width=2, marker_size=4):
    """Standard 2D CTD profile: <variable> vs. depth, surface at top.

    Unlike a glider, a CTD cast doesn't move through lon/lat \u2014 rendering it in 3D space
    (like a curtain) would misleadingly imply horizontal extent it doesn't actually have. This
    is the standard format for casts: where a cast was taken is shown by its marker on the map
    (outside this notebook), not by this plot, which focuses purely on the depth profile.
    """
    variable_label = variable_label or variable_col
    title = title or f"CTD Profile \u2014 {variable_label}"

    df_sorted = df.sort_values("Depth")
    lat, lon = df["Latitude"].iloc[0], df["Longitude"].iloc[0]
    ns, ew = ("N" if lat >= 0 else "S"), ("E" if lon >= 0 else "W")
    subtitle = f"Cast location: {abs(lat):.4f}\u00b0{ns}, {abs(lon):.4f}\u00b0{ew}"

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_sorted[variable_col], y=df_sorted["Depth"].abs(),
        mode="lines+markers",
        line=dict(width=line_width, color=line_color),
        marker=dict(size=marker_size, color=line_color),
        hovertemplate=f"{variable_label}: %{{x}}<br>Depth: %{{y}} m<extra></extra>",
        showlegend=False,
    ))
    fig.update_layout(
        title=f"{title}<br><sup>{subtitle}</sup>",
        xaxis_title=variable_label,
        yaxis_title="Depth (m)",
        yaxis=dict(autorange="reversed"),  # surface at top, standard oceanographic profile convention
    )
    return fig


## 8. Load data + plot

Toggle `USE_SAMPLE_DATA` / `USE_SAMPLE_BATHY` in `CONFIG` above to switch between demo and real data.

In [ ]:
# --- Bathymetry (shared basemap for both plots below) ---
bathy_cfg = CONFIG["BATHYMETRY"]
if bathy_cfg["USE_SAMPLE_BATHY"]:
    bathymetry = generate_sample_bathymetry()
else:
    bathymetry = load_bathymetry(bathy_cfg["DATA_PATH"], bathy_cfg["FILE_TYPE"], bathy_cfg["COLUMN_MAP"],
                                  depth_positive_down=bathy_cfg["DEPTH_POSITIVE_DOWN"])

print(f"Bathymetry: lon [{bathymetry['lon'].min():.3f}, {bathymetry['lon'].max():.3f}], "
      f"lat [{bathymetry['lat'].min():.3f}, {bathymetry['lat'].max():.3f}], "
      f"grid shape {bathymetry['depth'].shape}")



In [ ]:
# Sanity-check what this bathymetry file actually covers, before trusting it as the basemap
plot_bathymetry_extent(
    bathymetry,
    region_box=(CONFIG["REGION"]["lon_range"], CONFIG["REGION"]["lat_range"]),
    region_label="Barkley Sound (target)\noutside data coverage",
    mask_exact_zero=True,  # this file's cells north of ~51\u00b0N use 0.0 as a fill value, not real elevation
)


In [ ]:
# --- Glider track ---
glider_cfg = CONFIG["GLIDER"]
glider_var = glider_cfg["COLUMN_MAP"]["variable"]
if glider_cfg["USE_SAMPLE_DATA"]:
    glider_df = generate_sample_glider_data(variable_col=glider_var)
else:
    glider_df = load_platform_data(glider_cfg["DATA_PATH"], glider_cfg["FILE_TYPE"], glider_cfg["COLUMN_MAP"])
if glider_cfg["DEPTH_POSITIVE_DOWN"]:
    glider_df["Depth"] = -glider_df["Depth"].abs()

print(f"Glider: {len(glider_df)} rows. Columns: {list(glider_df.columns)}")

glider_fig = plot_glider_curtain(
    glider_df, variable_col=glider_var, variable_label=glider_cfg["VARIABLE_LABEL"],
    color_scale=glider_cfg["COLOR_SCALE"], marker_size=glider_cfg["MARKER_SIZE"],
    bathymetry=bathymetry if glider_cfg.get("USE_BATHYMETRY", True) else None,
    bathy_buffer_deg=bathy_cfg["BUFFER_DEG"],
    bathy_max_grid_size=bathy_cfg["MAX_GRID_SIZE"], bathy_colorscale=bathy_cfg["COLOR_SCALE"],
    bathy_opacity=bathy_cfg["OPACITY"],
)
glider_fig.show()


In [ ]:
# --- CTD cast ---
ctd_cfg = CONFIG["CTD"]
ctd_var = ctd_cfg["COLUMN_MAP"]["variable"]
if ctd_cfg["USE_SAMPLE_DATA"]:
    ctd_df = generate_sample_ctd_data(variable_col=ctd_var)
else:
    ctd_df = load_platform_data(ctd_cfg["DATA_PATH"], ctd_cfg["FILE_TYPE"], ctd_cfg["COLUMN_MAP"])
if ctd_cfg["DEPTH_POSITIVE_DOWN"]:
    ctd_df["Depth"] = -ctd_df["Depth"].abs()

print(f"CTD cast: {len(ctd_df)} rows. Columns: {list(ctd_df.columns)}")

ctd_fig = plot_ctd_profile(
    ctd_df, variable_col=ctd_var, variable_label=ctd_cfg["VARIABLE_LABEL"],
    line_color=ctd_cfg["LINE_COLOR"], marker_size=ctd_cfg["MARKER_SIZE"],
)
ctd_fig.show()
